In [ ]:
import gffutils
import pyfaidx
from Bio.Seq import Seq
import subprocess
from Bio import SeqIO

from Bio import Phylo

import pandas as pd
import json


In [ ]:
#prep work, split filtered fasta into pgcoe and rv, align both to reference, remove reference from aligned output

#for target in ['RSVA', 'RSVB']:
for target in []:
    fasta = f'inputs/{target}_filtered.fasta'
    pgcoerecs = []
    rvrecs = []
    for record in SeqIO.parse(fasta, "fasta"):
        record.id = record.id
        if("Yale-RSV" in record.id):
            record.id = record.id.replace("_", "-")
            pgcoerecs.append(record)
        elif("Yale-RV" in record.id):
            record.id = record.id.split("_")[0]
            rvrecs.append(record)
    SeqIO.write(pgcoerecs, fasta.replace("_filtered.fasta", "_pgcoe.fasta"), "fasta")
    SeqIO.write(rvrecs, fasta.replace("_filtered.fasta", "_rv.fasta"), "fasta")

    for out in ['pgcoe', 'rv']:
        gmafftf = fasta.replace("_filtered.fasta", f"_{out}_aligned.fasta")
        # Align sequences to reference
        with open(gmafftf, "w") as f: 
            subprocess.run(['mafft', '--auto', '--thread', '4', '--keeplength', '--add',
                            fasta.replace("_filtered.fasta", f"_{out}.fasta"), f'refs/{target}.fasta'], 
                            stdout=f, text=True)

        aligned_records = list(SeqIO.parse(gmafftf, "fasta"))
        filtered_records = aligned_records[1:] # Exclude the reference sequence (first record)
        # Overwrite gmafftf with filtered records (without the original reference)
        with open(gmafftf, "w") as out_handle:
            SeqIO.write(filtered_records, out_handle, "fasta")


In [ ]:
def extract_gene_sequences(fasta_path, gff_path, gene_name, nuc_out, aa_out):
    """
    Extract nucleotide and amino acid sequences for a specific gene from a multi-sequence FASTA and GFF3.
    Args:
        fasta_path (str): Path to input FASTA file (aligned or unaligned).
        gff_path (str): Path to GFF3 file.
        gene_name (str): Name of the gene to extract.
        nuc_out (str): Output path for nucleotide sequences.
        aa_out (str): Output path for amino acid sequences.
    """
    db = gffutils.create_db(gff_path, dbfn=':memory:')
    fasta = pyfaidx.Fasta(fasta_path)
    # Find the CDS feature for the gene
    cds_list = [cds for cds in db.features_of_type('CDS', order_by='start') if cds.attributes.get('gene', [''])[0] == gene_name]
    if not cds_list:
        raise ValueError(f"Gene {gene_name} not found in GFF3.")
    cds = cds_list[0]
    seqid = cds.seqid
    start = cds.start - 1  # pyfaidx uses 0-based indexing
    end = cds.end

    # Clear output files
    open(aa_out, "w").close()
    open(nuc_out, "w").close()

    with open(aa_out, "a") as faa, open(nuc_out, "a") as fnuc:
        for seq_record in list(fasta.keys()):
            nstops = [Seq(fasta[seq_record][(start+n):(end+n)].seq.replace('-', 'N')).translate().count('*') for n in [0,1,2]]
            n = nstops.index(min(nstops))
            nseq = fasta[seq_record][(start+n):(end+n)].seq
            aa = Seq(nseq.replace('-', 'N')).translate()
            faa.write(f">{seq_record}    {gene_name} (n={n})\n{aa}\n")
            fnuc.write(f">{seq_record}    {gene_name}\n{nseq.replace('-', 'N')}\n")



def filter_n_sequences(nuc_fasta, aa_fasta, nuc_out, aa_out, n_threshold=0.2):
    """
    Remove sequences from nucleotide and amino acid FASTA files if the nucleotide sequence has > n_threshold fraction of 'n'.
    Args:
        nuc_fasta (str): Path to nucleotide FASTA file.
        aa_fasta (str): Path to amino acid FASTA file.
        nuc_out (str): Output path for filtered nucleotide FASTA.
        aa_out (str): Output path for filtered amino acid FASTA.
        n_threshold (float): Fraction threshold for 'n' bases (default 0.2).
    """
    nuc_records = SeqIO.to_dict(SeqIO.parse(nuc_fasta, "fasta"))
    aa_records = SeqIO.to_dict(SeqIO.parse(aa_fasta, "fasta"))

    keep_ids = []
    drop_ids = []
    for rec_id, rec in nuc_records.items():
        seq = str(rec.seq).lower()
        n_frac = seq.count('n') / len(seq) if len(seq) > 0 else 1.0
        if n_frac <= n_threshold:
            keep_ids.append(rec_id)
        else:
            drop_ids.append(rec_id)

    with open(nuc_out, "w") as nf, open(aa_out, "w") as af:
        SeqIO.write([nuc_records[i] for i in keep_ids if i in nuc_records], nf, "fasta")
        SeqIO.write([aa_records[i] for i in keep_ids if i in aa_records], af, "fasta")
    return(len(keep_ids), len(drop_ids))

In [ ]:
def extract_reference_gene_aa(fasta_path, gff_path, prefix=""):
    """
    Extract reference amino acid sequences for each CDS from a reference fasta and gff,
    and write all to a single output file.
    Args:
        fasta_path (str): Path to reference FASTA file.
        gff_path (str): Path to GFF3 file.
        output_dir (str): Output directory for amino acid sequences.
    """
    ref_fasta = pyfaidx.Fasta(fasta_path)
    db = gffutils.create_db(gff_path, dbfn=':memory:')
    for cds in db.features_of_type('CDS', order_by='start'):
        gene = cds.attributes['gene'][0]
        with open(f"{prefix}{gene}_aa.fasta", "w") as f:
            seqid = cds.seqid
            start = cds.start - 1  # pyfaidx uses 0-based
            end = cds.end
            ref_seq = ref_fasta[seqid][start:end]
            nstops = [Seq(ref_seq[(n):(end-start+n)].seq.replace('-', 'N')).translate().count('*') for n in [0,1,2]]
            n = nstops.index(min(nstops))
            nseq = ref_seq[(n):(end-start+n)].seq
            aa = Seq(nseq.replace('-', 'N')).translate()
            f.write(f">{seqid}    {gene} (n={n})\n{aa}\n")


In [ ]:
# # get sequences for gene region for all consensus sequences
# # filter sequences with > 20% Ns

# for cds in db.features_of_type('CDS', order_by='start'):
#     gene = cds.attributes['gene'][0]
#     extract_gene_sequences(fastaf, gfff, gene, f"geneseqs/{gene}_sequences.fasta", f"geneseqs/{target}_{gene}_aa.fasta")

#     nkeep, ndrop = filter_n_sequences(f"geneseqs/{gene}_sequences.fasta", f"geneseqs/{target}_{gene}_aa.fasta", 
#                        f"geneseqs/{gene}_sequences_filt.fasta", f"geneseqs/{target}_{gene}_aa_filt.fasta", 
#                         n_threshold=0.2)
#     print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")


In [ ]:
def run_iqtree(iqprefix, codonf):
    """Run IQ-TREE on the codon alignment.""" 
    subprocess.run(["iqtree", "--prefix", iqprefix, "-s", codonf, 
                "-m", "GTR+G", "-nt", "AUTO", "-bb", "1000", "-alrt", "1000",
                "--redo"], check=True)


In [ ]:
def get_codon_alignment(gaafastaf, gfastaf, reffasta, gmafftf, codonf):
    """Generate a codon alignment using pal2nal from an amino acid alignment and the corresponding nucleotide sequences."""
    pal2nal = "/opt/miniconda3/envs/hyphy/bin/pal2nal.pl"

    #run mafft to align aa sequences
    command = ["mafft", '--keeplength',
               #'--localpair', '--maxiterate', '1000', '--op', '3.0',
               '--auto', '--op', '3.0',
               '--anysymbol', '--add', gaafastaf, reffasta]
    print(" ".join(command))
    with open(gmafftf, 'w') as outfile, open(gmafftf + ".err", 'w') as errfile:
        subprocess.run(command, stdout=outfile, stderr=errfile, check=True)

    # Parse MAFFT output and remove the original reference sequence from reffasta
    ref_id = list(SeqIO.parse(reffasta, "fasta"))[0].id
    aligned_records = list(SeqIO.parse(gmafftf, "fasta"))
    filtered_records = [rec for rec in aligned_records if rec.id != ref_id]
    # Overwrite gmafftf with filtered records (without the original reference)
    with open(gmafftf, "w") as out_handle:
        SeqIO.write(filtered_records, out_handle, "fasta")


    # Run pal2nal to generate codon alignment from the MAFFT output and the nucleotide fasta
    command = [pal2nal, gmafftf, gfastaf, "-nomismatch", "-nogap", "-output", "fasta"]
    print(" ".join(command))
    with open(codonf, "w") as outfile:
        subprocess.run(command, stdout=outfile, check=True)

def trim_stop_codons(codonf, codontrimf, mincodons=10):
    """Trim codon alignment to remove sequences after the first stop codon found in any sequence."""
    # Define stop codons
    STOP_CODONS = {"TAA", "TAG", "TGA"}

    records = list(SeqIO.parse(codonf, "fasta"))
    L = len(records[0].seq)

    n_codons = L // 3
    trim_at = n_codons  # default: keep full length
    checkcodons = min(mincodons, n_codons)  # check up to last 'mincodons'(10) codons

    # Look for stops in the last 'mincodons' codons
    for rec in records:
        codons = [str(rec.seq[i:i+3]).upper() for i in range(0, L, 3)]
        for offset in range(checkcodons, 0, -1):  # check from -checkcodons to -1 codons
            idx = n_codons - offset
            if idx < 0:
                continue
            if codons[idx] in STOP_CODONS:
                trim_at = min(trim_at, idx)  # earliest stop position
                break

    # Convert codon index back to nt length
    trim_nt = trim_at * 3
    print(f"Trimming alignment {L//3} -> {trim_at} codons ({trim_nt} nt).")

    with open(codontrimf, "w") as out:
        for rec in records:
            rec.seq = rec.seq[:trim_nt]
            SeqIO.write(rec, out, "fasta")


In [ ]:
#mafft  --auto --keeplength --add inputs/RSVA_all_aligned.fasta  refs/RSVA.fasta > inputs/RSVA_all_refaligned.fasta
#mafft  --auto --keeplength --add inputs/RSVB_all_aligned.fasta  refs/RSVB.fasta > inputs/RSVB_all_refaligned.fasta
#mafft  --auto --keeplength --add inputs/RSVA_filtered.fasta  refs/RSVA.fasta > inputs/RSVA_yaleonly_aligned.fasta
#mafft  --auto --keeplength --add inputs/RSVB_filtered.fasta  refs/RSVB.fasta > inputs/RSVB_yaleonly_aligned.fasta

targets = ["RSVB", "RSVA"]
for target in targets:
    print(f"Processing {target}")
    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    extract_reference_gene_aa(reff, gfff, f'./refgenes/{target}_')

db = gffutils.create_db(gfff, dbfn=':memory:')
geneids = [cds.attributes['gene'][0] for cds in db.features_of_type('CDS', order_by='start')]




In [ ]:
#PARSE CODON ALIGNMENTS FOR ALL SAMPLES

#genelist = ["G", "F"]
genelist = geneids
targets = ["RSVB", "RSVA"]
for target in targets:
    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    db = gffutils.create_db(gfff, dbfn=':memory:')

    fastaf = f'inputs/{target}_all_aligned.fasta'  # <-ns aligned output + bgr 
    
    print(f"Processing {target}")
    for gene in genelist:
        reffasta = f"refgenes/{target}_{gene}_aa.fasta"                #reference amino acid sequence

        #input sequences
        gfasta = f"geneseqs/{target}_{gene}_sequences.fasta"           #nucleotide sequences
        gaafasta = f"geneseqs/{target}_{gene}_aa.fasta"                #amino acid sequences

        #make codon alignment
        gmafftf = f"geneseqs/{target}_{gene}_aa_mafft.fasta"            #amino acid alignment
        codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

        #filter tree / sequences to match
        wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

        #hyphy outputs
        fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output


        #extract gene region for all samples, filter sequences with > 5% Ns
        extract_gene_sequences(fastaf, gfff, gene, gfasta, gaafasta)
        nkeep, ndrop = filter_n_sequences(gfasta, gaafasta, gfasta, gaafasta, n_threshold=0.05)
        print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")


        #get codon alignment, trim to last stop codon
        get_codon_alignment(gaafasta, gfasta, reffasta, gmafftf, codonf)
        trim_stop_codons(codonf, codonf, mincodons=20)
